<a href="https://colab.research.google.com/github/maker-arch/D2LExercises/blob/main/autogradqs_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import torch

x = torch.ones(5)
y = torch.zeros(3)
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

In [ ]:
print(f"gradient func for z is ", z.grad_fn)
print(f"gradient func for loss is ", loss.grad_fn)

gradient func for z is  <AddBackward0 object at 0x7dbe0a22c970>
gradient func for loss is  <BinaryCrossEntropyWithLogitsBackward0 object at 0x7dbe0a22c970>


In [ ]:
loss.backward()


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [ ]:
print(w.grad)
print(b.grad)

tensor([[0.0450, 0.0023, 0.2415],
        [0.0450, 0.0023, 0.2415],
        [0.0450, 0.0023, 0.2415],
        [0.0450, 0.0023, 0.2415],
        [0.0450, 0.0023, 0.2415]])
tensor([0.0450, 0.0023, 0.2415])


In [ ]:
z = torch.matmul(x, w) + b
print(z.requires_grad)

with torch.no_grad():
  z = torch.matmul(x, w) + b
  print(z.requires_grad)

True
False


In [ ]:
z = torch.matmul(x, w) + b
z_det = z.detach()
print(z_det.requires_grad)

False


In [12]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


In [14]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

# 获取训练数据集
training_data = datasets.FashionMNIST(
    root = "/data",
    train = True,
    download = True,
    transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

# 获取测试数据集
test_data = datasets.FashionMNIST(
    root = "/data",
    train = False,
    download = True,
    transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

# 准备Dataloader用于读取数据
train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

# 定义网络
class NeuralNetwork(nn.Module):
    def __init__(self):
      super().__init__()
      self.flatten = nn.Flatten()
      self.linear_relu_stack = nn.Sequential(
          nn.Linear(28*28, 512),
          nn.ReLU(),
          nn.Linear(512, 512),
          nn.ReLU(),
          nn.Linear(512, 10)
      )

    def forward(self, x):
      x = self.flatten(x)
      logits = self.linear_relu_stack(x)
      return logits

model = NeuralNetwork()
print(model)

100%|██████████| 26.4M/26.4M [00:01<00:00, 14.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 230kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 4.28MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 21.5MB/s]

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [15]:
loss_fn = nn.CrossEntropyLoss()

In [16]:
learning_rate = 1e-3
batch_size = 64
epochs = 5
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [40]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X,y) in enumerate(dataloader):
        y_pred = model(X)
        loss =loss_fn(y_pred, y)
        loss.backward()
        for idx, param in enumerate(model.parameters()):
            # 只打印非空、有数值的参数，取第一个元素
            if param.numel() > 0:
                print(f"参数{idx}首值: {param.flatten()[0].item()}")
        optimizer.step()
        for idx, param in enumerate(model.parameters()):
        # 只打印非空、有数值的参数，取第一个元素
            if param.numel() > 0:
                print(f"参数{idx}更新后值: {param.flatten()[0].item()}")

        optimizer.zero_grad()

        # if batch % 100 == 0:
        #     loss, current = loss.item(), batch * batch_size + len(X)
        #     print(f"loss: {loss:>7f}, [current: {current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batch = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batch
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [38]:
for batch, (X,y) in enumerate(train_dataloader):
  print(X.shape)
  print(y.shape)
  y_pred = model(X)
  print(y_pred.shape)
  loss =loss_fn(y_pred, y)
  loss.backward()
  for idx, param in enumerate(model.parameters()):
    # 只打印非空、有数值的参数，取第一个元素
    if param.numel() > 0:
        print(f"参数{idx}首值: {param.flatten()[0].item()}")

  optimizer.step()
  for idx, param in enumerate(model.parameters()):
    # 只打印非空、有数值的参数，取第一个元素
    if param.numel() > 0:
        print(f"参数{idx}更新后值: {param.flatten()[0].item()}")

  break


torch.Size([64, 1, 28, 28])
torch.Size([64])
torch.Size([64, 10])
参数0首值: -0.033354081213474274
参数1首值: 0.036771584302186966
参数2首值: 0.0486774742603302
参数3首值: 0.015543410554528236
参数4首值: -0.0292214285582304
参数5首值: -0.01694207265973091
参数0更新后值: -0.033354081213474274
参数1更新后值: 0.036777909845113754
参数2更新后值: 0.04867837578058243
参数3更新后值: 0.01555210817605257
参数4更新后值: -0.029232487082481384
参数5更新后值: -0.016902584582567215


In [41]:
epochs = 2

for i in range(epochs):
    print(f"Epoch {i+1}\n-----------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    # test_loop(test_dataloader, model, loss_fn)
print("Done!")

流式输出内容被截断，只能显示最后 5000 行内容。
参数5首值: -0.013347598724067211
参数0更新后值: -0.03335420787334442
参数1更新后值: 0.038035158067941666
参数2更新后值: 0.04913840815424919
参数3更新后值: 0.015965357422828674
参数4更新后值: -0.030698008835315704
参数5更新后值: -0.013328351080417633
参数0首值: -0.03335420787334442
参数1首值: 0.038035158067941666
参数2首值: 0.04913840815424919
参数3首值: 0.015965357422828674
参数4首值: -0.030698008835315704
参数5首值: -0.013328351080417633
参数0更新后值: -0.03335420787334442
参数1更新后值: 0.038035254925489426
参数2更新后值: 0.04913836345076561
参数3更新后值: 0.015964774414896965
参数4更新后值: -0.030695542693138123
参数5更新后值: -0.01330674160271883
参数0首值: -0.03335420787334442
参数1首值: 0.038035254925489426
参数2首值: 0.04913836345076561
参数3首值: 0.015964774414896965
参数4首值: -0.030695542693138123
参数5首值: -0.01330674160271883
参数0更新后值: -0.03335420787334442
参数1更新后值: 0.03803328052163124
参数2更新后值: 0.04913622513413429
参数3更新后值: 0.01596270501613617
参数4更新后值: -0.030693015083670616
参数5更新后值: -0.013324129395186901
参数0首值: -0.03335420787334442
参数1首值: 0.03803328052163124
参数2首值: 0.049